# OpenTopography Point Elevation API
## Comparing Elevation Datasets at a Single Location

This notebook demonstrates how to use the **OpenTopography Point Elevation API** to retrieve elevation values from multiple global and regional datasets at a single geographic coordinate, and visualize how those values compare.

### What the API does
Given a longitude, latitude, and a dataset shortname, the API returns the interpolated elevation at that point — without the overhead of downloading and processing a full DEM. This makes it efficient for use cases like:

- Annotating a list of addresses or GPS waypoints with elevation
- Rapid cross-dataset comparison at specific field sites
- Validating or correcting elevation values in tabular data

**API documentation:** https://portal.opentopography.org/apidocs/#/Public/getPointElevation

**Register for a free API key:** https://portal.opentopography.org/newUser

## 1. Installation

Install required packages if you haven't already:

```
pip install requests matplotlib numpy contextily geopandas shapely
```

> **Note:** `contextily` and `geopandas` are only used for the map panels in the final figure.
> If you only want the elevation bar charts, those two packages are not required.

In [ ]:
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import contextily as ctx
import geopandas as gpd
from shapely.geometry import Point, box
from matplotlib.gridspec import GridSpec

## 2. Configuration

Set your API key and the endpoint URL. An API key is required.  
Once registered with your free OpenTopography account you can get your API key here:
```
https://portal.opentopography.org/requestService?service=api
```

The base URL for the Point Elevation API is:
```
https://portal.opentopography.org/API/v1/elevation
```

A complete request looks like:
```
GET /API/v1/elevation?longitude=-104.99&latitude=39.74&dataset=COP30&API_Key=YOUR_KEY
```

In [ ]:
# Replace with your own API key from https://portal.opentopography.org/requestAPIKey
API_KEY  = "YOUR_API_KEY_HERE"

# Point Elevation API endpoint
BASE_URL = "https://portal.opentopography.org/API/v1/elevation"

## 3. Datasets and Vertical Datums

OpenTopography hosts many global and regional elevation datasets, each referenced to a **vertical datum** — the surface from which elevations are measured. Understanding the datum is essential for making meaningful comparisons between datasets. Most global datasets use one of a handful of standardized geoid models (EGM96, EGM2008) or the WGS84 ellipsoid, but regional datasets often adopt nationally-defined datums tailored to local gravity conditions, such as NAVD88 for North America or country-specific geoids for New Zealand, Switzerland, and Norway.

### The main datum families

| Datum | Description |
|---|---|
| **EGM96 / EGM2008** (geoid) | Heights above a model of Earth's gravitational equipotential surface approximating mean sea level. What most people think of as "elevation above sea level." |
| **WGS84 Ellipsoid** | Heights above the smooth mathematical ellipsoid used by GPS. Purely geometric — can differ from orthometric (geoid) heights by tens of meters. |
| **NAVD88** | The official vertical datum for the contiguous US, derived from a network of tide gauges and precise leveling surveys. |

The API returns the vertical datum for each query in the `VCRS_EPSG` field of the response,
which is the EPSG code for the dataset's vertical coordinate reference system.

In [ ]:
# Group datasets by their vertical datum.
# Only datasets with global or near-global coverage are included here;
# regional datasets (Arctic, Antarctic, Canada, New Zealand, etc.) could be
# added for locations within their coverage areas.  Use the dataset "shortname"
# as listed on each dataset's landing page.
DATUM_GROUPS = {
    "EGM2008 Geoid":   ["COP30", "COP90", "EU_DTM"],
    "EGM96 Geoid":     ["SRTM_GL1", "SRTM_GL3", "NASADEM", "AW3D30",
                        "SRTM15Plus", "GEBCOIceTopo", "GEBCOSubIceTopo"],
    "WGS84 Ellipsoid": ["GEDI_L3", "AW3D30_E", "SRTM_GL1_Ellip", "GEDTM30"],
    "NAVD88":          ["USGS10m", "USGS30m"],
}

# Flat lookup: dataset shortname -> datum name
DATASET_DATUM = {
    ds: datum
    for datum, datasets in DATUM_GROUPS.items()
    for ds in datasets
}

# Color for each datum family — used consistently in bar charts and the legend
DATUM_COLORS = {
    "EGM2008 Geoid":   "#2196F3",  # blue
    "EGM96 Geoid":     "#4CAF50",  # green
    "WGS84 Ellipsoid": "#FF9800",  # orange
    "NAVD88":          "#9C27B0",  # purple
}

# All datasets to query, in a consistent display order
DATASET_ORDER = [
    "COP30", "COP90",
    "SRTM_GL1", "SRTM_GL1_Ellip", "SRTM_GL3",
    "NASADEM",
    "AW3D30", "AW3D30_E",
    "GEDI_L3",
    "GEDTM30",
    "SRTM15Plus",
    "GEBCOIceTopo", "GEBCOSubIceTopo",
    "EU_DTM",
    "USGS10m", "USGS30m",
]

# Left-to-right order of datum groups in bar charts
DATUM_ORDER = ["EGM2008 Geoid", "EGM96 Geoid", "WGS84 Ellipsoid", "NAVD88"]


def sorted_by_datum(datasets):
    """Return datasets ordered by datum group, then alphabetically within each group."""
    def sort_key(ds):
        datum = DATASET_DATUM.get(ds, "")
        try:
            return (DATUM_ORDER.index(datum), ds)
        except ValueError:
            return (len(DATUM_ORDER), ds)  # unknown datums go last
    return sorted(datasets, key=sort_key)

## 4. Querying the API

The API accepts four query parameters and returns a JSON object:

| Parameter | Description |
|---|---|
| `longitude` | WGS84 decimal degrees (negative = west) |
| `latitude` | WGS84 decimal degrees (negative = south) |
| `dataset` | Dataset shortname (e.g. `COP30`, `USGS10m`) |
| `API_Key` | Your OpenTopography API key |

**Example response** for `COP30` at Denver, CO:
```json
{
  "Elevation": "1608.79",
  "Shortname": "COP30",
  "VCRS_WKT": "VERTCRS[\"EGM2008 height\", ...]",
  "VCRS_EPSG": "3855",
  "Unit": "Meters"
}
```

The `VCRS_EPSG` field is the EPSG code for the dataset's vertical CRS, which is useful
if you need to programmatically transform between datums.

In [ ]:
def query_elevation(longitude, latitude, dataset):
    """
    Retrieve the interpolated elevation at a single point from one OT dataset.

    Parameters
    ----------
    longitude : float
        WGS84 decimal degrees. Negative values are west of the prime meridian.
    latitude : float
        WGS84 decimal degrees. Negative values are south of the equator.
    dataset : str
        OpenTopography dataset shortname, e.g. "COP30" or "USGS10m".

    Returns
    -------
    float or None
        Elevation in meters relative to the dataset's vertical datum,
        or None if the point has no data or is outside the dataset's coverage.
    """
    params = {
        "longitude": longitude,
        "latitude":  latitude,
        "dataset":   dataset,
        "API_Key":   API_KEY,
    }

    response = requests.get(BASE_URL, params=params, timeout=30)

    # 404: point has no data in this dataset (NoData raster value)
    # 422: point is outside the spatial bounds of the dataset
    # Both are expected for regional datasets queried at out-of-coverage locations.
    if response.status_code in (404, 422):
        return None

    response.raise_for_status()  # raises HTTPError for other 4xx / 5xx responses

    data = response.json()
    elev = data.get("Elevation")

    # The API sets Elevation to None when no valid value exists at this point
    if elev is None:
        return None

    return float(elev)

In [ ]:
def query_all_datasets(longitude, latitude):
    """
    Query every dataset in DATASET_ORDER for the given location.

    Returns a dict {dataset_shortname: elevation_float} containing only
    datasets that returned a valid elevation. Datasets outside their
    coverage area are silently omitted.
    """
    elevations = {}

    for dataset in DATASET_ORDER:
        try:
            elev = query_elevation(longitude, latitude, dataset)
        except Exception as e:
            print(f"  {dataset:<20}  warning: {e}")
            elev = None

        if elev is not None:
            print(f"  {dataset:<20}  {elev:.2f} m")
            elevations[dataset] = elev
        else:
            print(f"  {dataset:<20}  (no data)")

    return elevations

## 5. Define Test Locations

We compare three terrain types to show how dataset differences vary by environment:

| Location | Terrain type | Why interesting |
|---|---|---|
| Denver, CO | Urban | Mix of built surface and bare earth; USGS datasets available for comparison |
| Saharan Reg, Algeria | Arid desert | Flat, sparse vegetation; radar and lidar should agree closely |
| Black Forest, Germany | Dense forest | Canopy penetration differences between radar (SRTM) and lidar-derived products are visible |

Each scenario also specifies zoom levels for the two map panels in the figure:
- `overview_buffer_deg`: half-width of the regional context map (degrees)
- `detail_buffer_deg`: half-width of the zoomed-in satellite imagery (degrees)

In [ ]:
SCENARIOS = [
    {
        "label":       "Urban",
        "description": "Denver, CO",
        "longitude":   -104.985462,
        "latitude":      39.739232,
        "overview_buffer_deg": 8.0,   # degrees of lat/lon around the point for the context map
        "detail_buffer_deg":   0.015, # degrees for the zoomed satellite imagery
    },
    {
        "label":       "Arid Desert",
        "description": "Saharan Reg, Algeria",
        "longitude":    2.5000,
        "latitude":    27.2000,
        "overview_buffer_deg": 8.0,
        "detail_buffer_deg":   0.05,
    },
    {
        "label":       "Forested",
        "description": "Black Forest, Germany",
        "longitude":    8.1200,
        "latitude":    48.0150,
        "overview_buffer_deg": 6.0,
        "detail_buffer_deg":   0.02,
    },
]

## 6. Fetch Elevation Data

The cell below makes one API call per dataset per scenario.
With 16 datasets and 3 scenarios that is **48 API calls** — typically completes in about 60 seconds.

Results are collected in `scenarios_data`, a list of `(scenario_dict, elevations_dict)` tuples
that the visualization functions will consume.

In [ ]:
scenarios_data = []  # list of (scenario_dict, elevations_dict) tuples

for scenario in SCENARIOS:
    lat = scenario["latitude"]
    lon = scenario["longitude"]
    print(f"\n{scenario['label']} — {scenario['description']}")
    print(f"Location: {abs(lat):.4f}{'N' if lat >= 0 else 'S'}, "
          f"{abs(lon):.4f}{'E' if lon >= 0 else 'W'}")
    print("-" * 50)

    elevations = query_all_datasets(lon, lat)
    scenarios_data.append((scenario, elevations))

    print(f"  -> {len(elevations)} of {len(DATASET_ORDER)} datasets returned valid elevations")

print("\nAll queries complete.")

## 7. Visualization

The figure has one row per scenario, each with three panels:

| Panel | Content |
|---|---|
| **Left** | Vertical bar chart — elevation from each dataset, bars grouped and colored by vertical datum |
| **Middle** | Regional street map showing geographic context |
| **Right** | Zoomed-in satellite imagery centered on the query point |

Bars within the same datum family are grouped together (with a small gap between groups)
so you can quickly see whether datum offsets explain elevation differences.

In [ ]:
def project_to_web_mercator(longitude, latitude, buffer_deg):
    """
    Build a bounding box and point in Web Mercator (EPSG:3857) for use with
    contextily basemap tiles.

    Returns (bounds_array, point_gdf) where bounds is [xmin, ymin, xmax, ymax].
    """
    lon_min, lon_max = longitude - buffer_deg, longitude + buffer_deg
    lat_min, lat_max = latitude  - buffer_deg, latitude  + buffer_deg

    # Bounding box in WGS84, projected to Web Mercator
    bbox = gpd.GeoDataFrame(
        {"geometry": [box(lon_min, lat_min, lon_max, lat_max)]},
        crs="EPSG:4326",
    ).to_crs(epsg=3857)

    # Query point in WGS84, projected to Web Mercator
    pt = gpd.GeoDataFrame(
        {"geometry": [Point(longitude, latitude)]},
        crs="EPSG:4326",
    ).to_crs(epsg=3857)

    return bbox.total_bounds, pt


def add_map_panel(ax, longitude, latitude, buffer_deg, provider, marker_size=120):
    """
    Render a tiled basemap on ax with the query point marked as a red star.

    Uses contextily to fetch tiles from the given provider.
    ax.scatter() is used instead of geopandas .plot() to avoid
    geopandas forcing aspect='equal', which causes whitespace issues.
    """
    bounds, pt = project_to_web_mercator(longitude, latitude, buffer_deg)

    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])

    # zoom="auto" lets contextily choose an appropriate tile zoom level
    ctx.add_basemap(ax, source=provider, zoom="auto", reset_extent=False)

    pt_x = pt.geometry.iloc[0].x
    pt_y = pt.geometry.iloc[0].y
    ax.scatter([pt_x], [pt_y], color="red", s=marker_size, marker="*",
               zorder=10, edgecolors="white", linewidths=0.8)

    ax.set_aspect("auto")
    ax.set_xticks([])
    ax.set_yticks([])


def make_overview_panel(ax, longitude, latitude, buffer_deg):
    """Middle panel: zoomed-out street map for regional geographic context."""
    add_map_panel(ax, longitude, latitude, buffer_deg,
                  provider=ctx.providers.Esri.WorldStreetMap,
                  marker_size=80)
    ax.set_title("Regional context", fontsize=10, color="#333333", pad=4)


def make_detail_panel(ax, longitude, latitude, buffer_deg):
    """Right panel: zoomed-in satellite imagery with coordinate label."""
    add_map_panel(ax, longitude, latitude, buffer_deg,
                  provider=ctx.providers.Esri.WorldImagery,
                  marker_size=150)
    hem_ns = "N" if latitude  >= 0 else "S"
    hem_ew = "E" if longitude >= 0 else "W"
    coord_str = f"{abs(latitude):.4f}{hem_ns}, {abs(longitude):.4f}{hem_ew}"
    ax.set_title(coord_str, fontsize=8, color="white",
                 bbox=dict(facecolor="black", alpha=0.6, pad=2), pad=4)

In [ ]:
def make_elevation_panel(ax, scenario, elevations):
    """
    Left panel: vertical bar chart showing elevation from each dataset.

    Bars are grouped and color-coded by vertical datum, with a small gap
    between datum families so the grouping is visually clear.
    A dashed horizontal line marks the mean elevation across all datasets.
    """
    # Only include datasets that returned data, sorted by datum group
    datasets = sorted_by_datum([ds for ds in DATASET_ORDER if ds in elevations])
    values   = [elevations[ds] for ds in datasets]
    colors   = [DATUM_COLORS.get(DATASET_DATUM.get(ds, ""), "#888888") for ds in datasets]

    if not values:
        ax.text(0.5, 0.5, "No data returned for this location",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=9, color="gray")
        ax.set_title(f"{scenario['label']}  \u2014  {scenario['description']}",
                     fontsize=10, fontweight="bold", loc="left")
        return

    # Build x-positions with an extra gap whenever the datum group changes.
    # BAR_GAP creates the visual cluster between datum families.
    BAR_GAP    = 0.6
    x_pos      = []
    x          = 0
    prev_datum = None

    for ds in datasets:
        datum = DATASET_DATUM.get(ds, "")
        if prev_datum is not None and datum != prev_datum:
            x += BAR_GAP  # insert gap between datum groups
        x_pos.append(x)
        x += 1
        prev_datum = datum

    ax.bar(x_pos, values, color=colors, edgecolor="white", linewidth=0.4, width=0.7)

    val_min, val_max = min(values), max(values)
    val_range = val_max - val_min if val_max != val_min else 1.0

    # Elevation value label rotated above each bar
    for xp, val in zip(x_pos, values):
        ax.text(xp, val + val_range * 0.02, f"{val:.1f}",
                ha="center", va="bottom", fontsize=9, color="black",
                rotation=90, fontweight="bold")

    # Dashed horizontal line at the cross-dataset mean elevation
    mean_val = np.mean(values)
    ax.axhline(mean_val, color="black", linestyle="--", linewidth=1, alpha=0.5)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(datasets, fontsize=8, rotation=45, ha="right")
    ax.set_ylabel("Elevation (m)", fontsize=9)
    ax.set_title(
        f"{scenario['label']}  \u2014  {scenario['description']}\n"
        f"Mean elevation: {mean_val:.1f} m  (dashed line)",
        fontsize=10, fontweight="bold", loc="left", linespacing=1.6,
    )

    # Start the y-axis below the minimum so inter-dataset differences are visible
    padding_bottom = max(10.0, val_range * 0.5)
    padding_top    = max(5.0,  val_range * 0.3) + val_range * 0.15
    ax.set_ylim(bottom=val_min - padding_bottom, top=val_max + padding_top)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4, linewidth=0.5)

In [ ]:
def make_figure(scenarios_data, output_path="elevation_comparison.png"):
    """
    Assemble the full multi-panel comparison figure and save it to disk.

    Layout: one row per scenario, three columns:
      col 0 — elevation bar chart (grouped by datum)
      col 1 — regional street map
      col 2 — zoomed satellite imagery
    """
    n_rows = len(scenarios_data)
    fig    = plt.figure(figsize=(20, 5.5 * n_rows))
    fig.patch.set_facecolor("#F8F8F8")

    # GridSpec: bar chart column is slightly narrower than the two map columns
    gs = GridSpec(
        n_rows, 3, figure=fig,
        width_ratios=[0.8, 1.0, 1.0],
        hspace=0.50,
        wspace=0.03,        # tight spacing between map panels
        left=0.08, right=0.98,
        top=0.93,  bottom=0.10,
    )

    row_axes = []
    for row_idx, (scenario, elevations) in enumerate(scenarios_data):
        ax_elev     = fig.add_subplot(gs[row_idx, 0])
        ax_overview = fig.add_subplot(gs[row_idx, 1])
        ax_detail   = fig.add_subplot(gs[row_idx, 2])

        make_elevation_panel(ax_elev, scenario, elevations)
        make_overview_panel(ax_overview,
                            scenario["longitude"], scenario["latitude"],
                            scenario["overview_buffer_deg"])
        make_detail_panel(ax_detail,
                          scenario["longitude"], scenario["latitude"],
                          scenario["detail_buffer_deg"])
        row_axes.append(ax_elev)

    # Thin horizontal separator lines between rows
    fig.canvas.draw()  # force layout calculation so get_position() is accurate
    for i in range(len(row_axes) - 1):
        y_bottom = row_axes[i].get_position().y0
        y_top    = row_axes[i + 1].get_position().y1
        y_line   = ((y_bottom + y_top) / 2) - 0.01
        fig.add_artist(plt.Line2D(
            [0.08, 0.98], [y_line, y_line],
            transform=fig.transFigure,
            color="#AAAAAA", linewidth=1.2, linestyle="-", zorder=10,
        ))

    # Datum color legend along the bottom of the figure
    legend_handles = [
        mpatches.Patch(color=color, label=datum)
        for datum, color in DATUM_COLORS.items()
    ]
    fig.legend(
        handles=legend_handles,
        title="Vertical Datum:",
        title_fontsize=16, fontsize=14,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.0),
        ncol=len(DATUM_COLORS),
        framealpha=0.9, edgecolor="#CCCCCC",
    )

    fig.suptitle(
        "OpenTopography Point Elevation API \u2014 Dataset Comparison",
        fontsize=14, fontweight="bold", y=0.975,
    )

    plt.savefig(output_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    print(f"Figure saved to: {output_path}")
    plt.show()


# Generate the figure — displays inline in the notebook and saves to disk
make_figure(scenarios_data, output_path="elevation_comparison.png")

## 8. Interpreting the Results

### Why do elevations differ between datasets at the same point?

Several factors drive the spread you see in the bar charts:

**1. Vertical datums** — WGS84 ellipsoid heights (orange bars) are typically
10–50 m higher than geoid-based heights at mid-latitudes because the geoid sits below
the ellipsoid in most locations.

**2. Sensor type** — Radar-derived datasets (SRTM, COP30) partially penetrate vegetation
and return echoes near the top of the canopy. Lidar-derived datasets can filter out
vegetation returns to model bare earth. In forested terrain the difference can be
several meters.

**3. DSM vs DTM** — A Digital Surface Model (DSM) represents the first surface the sensor
sees (treetops, rooftops), while a Digital Terrain Model (DTM) represents the bare ground
beneath. In urban or forested areas a DSM and DTM for the same location can differ by
many meters.

**4. Spatial resolution** — A 30 m pixel and a 1 m pixel at the same coordinate are not
sampling the same thing. The coarser product averages over a much larger area, smoothing
out local relief. In complex terrain this can produce substantial differences.

---

## Summary

This notebook demonstrated how to use the OpenTopography Point Elevation API to:

- Query a single geographic coordinate across multiple global elevation datasets with a simple HTTP GET request
- Understand how vertical datum differences (EGM96, EGM2008, WGS84 ellipsoid, NAVD88) affect the returned elevation values
- Visualize cross-dataset elevation spread and see how terrain type (urban, desert, forest) influences agreement between datasets

The API is well-suited for any workflow that needs point elevations at scale — annotating address lists,
validating field survey data, or comparing datasets at specific sites — without the overhead of
downloading and processing full DEMs.

### API usage limits

During the initial release of the Point Elevation API, the following daily query limits apply:

| User type | Daily limit |
|---|---|
| Registered OpenTopography user | 50 queries/day |
| [OT+](https://opentopography.org/plus) member or academic user (`.edu` email) | 250 queries/day |

If your workflow requires higher volumes, consider the [OT+ program](https://opentopography.org/plus)
or contact OpenTopography to discuss your use case.